# 02 · Preprocessing
**Owner:** Laiba Shaikh · **Goal:** decide, with evidence, how documents and queries are prepared for each retriever, and export clean artifacts that every later notebook (and the API) reuses.

Principle: **sparse models need text normalisation; dense models do not.** Transformers use word order and function words, so for MiniLM/BGE we only collapse whitespace.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))          # so `import retrieval` works from notebooks/
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from retrieval.data import load_corpus, load_claims, build_qrels
FIG = ROOT / "results" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from retrieval.sparse import TfidfRetriever
from retrieval.evaluate import evaluate_run

## 1 · Building the document text

In [2]:
raw = [json.loads(l) for l in open(ROOT / "data" / "corpus.jsonl")]
corpus = load_corpus()
print("RAW      :", {k: (v if k != "abstract" else f"[{len(v)} sentences]") for k, v in raw[0].items() if k != "structured"})
print("PROCESSED:", corpus[0]["text"][:300], "...")
print()
texts = [d["text"] for d in corpus]
print("duplicate document texts:", len(texts) - len(set(texts)))
print("docs with a sentence that is only whitespace in the raw data:",
      sum(any(not s.strip() for s in r["abstract"]) for r in raw))
print("docs whose title and abstract were re-joined:", len(corpus))

RAW      : {'doc_id': 4983, 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'abstract': '[14 sentences]'}
PROCESSED: Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging. Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line sca ...

duplicate document texts: 0
docs with a sentence that is only whitespace in the raw data: 0
docs whose title and abstract were re-joined: 5183


Each document is indexed as **`title. sentence1 sentence2 …`**. The title carries high-value terms (disease/gene names), and the same string is used for TF-IDF, BM25, MiniLM and BGE so the comparison between models is fair — the *only* variable is the model.

## 2 · Tokenisation on biomedical text

In [3]:
sample = "IL-6 and p53 in 5-HT receptor, BRCA1/2 and 10-20% of CD4+ cells"
default = TfidfVectorizer().build_analyzer()
hyphen = TfidfVectorizer(token_pattern=r"(?u)\b[\w-]{2,}\b").build_analyzer()
print("default sklearn :", default(sample))
print("keep-hyphens    :", hyphen(sample))

default sklearn : ['il', 'and', 'p53', 'in', 'ht', 'receptor', 'brca1', 'and', '10', '20', 'of', 'cd4', 'cells']
keep-hyphens    : ['il-6', 'and', 'p53', 'in', '5-ht', 'receptor', 'brca1', 'and', '10-20', 'of', 'cd4', 'cells']


The default pattern splits `IL-6` into `il` (it drops the 1-character `6`). That *looks* harmful for biomedical text, so we test it instead of assuming — on **train** claims only:

In [4]:
tr = load_claims("train"); qr = build_qrels(tr)
queries = {str(c["id"]): c["claim"] for c in tr if str(c["id"]) in qr}
ids = [d["id"] for d in corpus]
variants = {
    "default token pattern":        dict(),
    "keep hyphenated tokens":       dict(token_pattern=r"(?u)\b[\w-]{2,}\b"),
    "single chars allowed":         dict(token_pattern=r"(?u)\b\w+\b"),
    "no stop-word removal":         dict(stop_words=None),
    "no lowercasing":               dict(lowercase=False),
}
rows = []
for name, kw in variants.items():
    r = TfidfRetriever(ids, texts, **kw)
    m = evaluate_run(r.run(queries, 100), qr)
    rows.append({"variant": name, "vocab": len(r.vec.vocabulary_), "nDCG@10": m["nDCG@10"], "Recall@10": m["Recall@10"], "Recall@100": m["Recall@100"]})
res = pd.DataFrame(rows).round(4)
print(res.to_string(index=False))

               variant  vocab  nDCG@10  Recall@10  Recall@100
 default token pattern  35617   0.8078     0.9208      0.9748
keep hyphenated tokens  47286   0.7945     0.9120      0.9698
  single chars allowed  35670   0.8074     0.9208      0.9748
  no stop-word removal  35899   0.8093     0.9201      0.9777
        no lowercasing  43652   0.7508     0.8850      0.9688


**Decision:** keep sklearn's default tokenizer and lowercasing. Findings:
* The hyphen-preserving tokenizer is *not* better (0.795 vs 0.808 nDCG@10), plausibly because claims paraphrase compounds, so exact tokens like `il-6` match less often than their pieces. A negative result worth reporting.
* **Lowercasing matters** (−0.057 nDCG@10 without it).
* **Stop-word removal makes no meaningful difference** (the gap is ~0.002, noise-level), so it stays a tuned hyperparameter in notebook 03 rather than a fixed decision.

## 3 · Preprocessing for dense models — deliberately minimal

In [5]:
claims = load_claims("dev")
print("raw claim      :", claims[3]["claim"])
print("whitespace-only:", re.sub(r"\s+", " ", claims[3]["claim"]).strip())
words = pd.Series([len(t.split()) for t in texts])
print(f"docs longer than 256 words: {(words > 256).mean():.1%}  |  longer than 400 words: {(words > 400).mean():.1%}")

raw claim      : 5% of perinatal mortality is due to low birth weight.
whitespace-only: 5% of perinatal mortality is due to low birth weight.
docs longer than 256 words: 26.8%  |  longer than 400 words: 2.5%


For MiniLM / BGE we do **not** lowercase, stem or remove stop-words (it would damage the contextual signal these models were trained on). Truncation is handled by the model's `max_seq_length`
(256 for MiniLM, 512 for BGE-small) — see `retrieval/dense.py::DENSE_MODELS`.

## 4 · Export artifacts (consumed by notebooks 03–07 and the evaluation phase)

In [ ]:
out = ROOT / "data" / "processed"; out.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{"doc_id": d["id"], "title": d["title"], "text": d["text"]} for d in corpus]).to_csv(out / "corpus_processed.csv", index=False)
for split in ["train", "dev", "test"]:
    cl = load_claims(split)
    qrels = build_qrels(cl)
    (out / f"queries_{split}.json").write_text(json.dumps({str(c["id"]): c["claim"] for c in cl}))
    if split != "test":
        (out / f"qrels_{split}.json").write_text(json.dumps({q: sorted(v) for q, v in qrels.items()}))
    print(split, "| queries:", len(cl), "| scorable (have qrels):", len(qrels))
print(sorted(p.name for p in out.iterdir()))

train | queries: 809 | scorable (have qrels): 505
dev | queries: 300 | scorable (have qrels): 188
test | queries: 300 | scorable (have qrels): 0
['corpus_processed.csv', 'qrels_dev.json', 'qrels_train.json', 'queries_dev.json', 'queries_test.json', 'queries_train.json']


### Summary of preprocessing decisions
| Step | TF-IDF / BM25 | MiniLM / BGE |
|---|---|---|
| Text unit | `title. abstract` | `title. abstract` |
| Whitespace collapse | ✅ | ✅ |
| Lowercase | ✅ (matters) | ❌ |
| English stop-words | tuned in nb 03 (no real effect) | ❌ |
| Custom biomedical tokenizer | ❌ (tested, not better) | n/a |
| Truncation | none | model `max_seq_length` |
| Query prefix | none | BGE only (tested in notebook 05) |